(structuring-data)= 
# Structuring Data

### Input Dataset Requirements

**Core Requirements (Flexible):**
- **X values**: Predictor variables (in below example: `'site'`, `'age'`, `'sex'`)
- **Y values**: Outcome variables (feature columns like `'y_thickness1'`, `'y_thickness2'`, etc.)

**Optional Requirements:**
- **Modality column**: Only needed if you want to identify different conditions/groups
  - Column name is flexible (below used `'affected'` as an example)
  - Healthy subjects should be labeled as `'None'`
  - Conditions can be any descriptive names (`'Schizo'`, `'Dementia'`, `'MDD'`, etc.)

**Key Insights:**
1. The documentation example shows a specific neuroimaging use case, but the package is more general-purpose
2. The minimum requirement is just predictors (X) and outcomes (Y)
3. The `'affected'` column is essentially for stratification/group analysis, not a core modeling requirement
4. This makes the package applicable to various domains beyond neuroimaging

### Generating Example Data

In [2]:
from IPython.display import display

import numpy as np
import pandas as pd
import random
import string

# Example data generation code
# Function to generate random site ID
def generate_random_site_id(length, seed):
    random.seed(seed)
    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for _ in range(length))

# Generate train dataset
np.random.seed(42)
n_train = 1000
train_sites_seeds = [1, 2, 3, 4, 5]  # Fixed seeds for reproducibility
train_sites = [generate_random_site_id(5, seed) for seed in train_sites_seeds]
age_train = np.random.randint(18, 80, n_train)
sex_train = np.random.choice([0, 1], size=n_train)
feature_volume_train1 = np.random.uniform(800, 1200, n_train)
feature_volume_train2 = np.random.uniform(800, 1200, n_train)
feature_volume_train3 = np.random.uniform(800, 1200, n_train)
feature_volume_train4 = np.random.uniform(800, 1200, n_train)
feature_volume_train5 = np.random.uniform(800, 1200, n_train)
# Non-linear age effect: values decline steeply through early adulthood and
# then flatten out at older ages (exponential decay) -- the trajectory seen
# in real cortical-thickness data. The two sexes follow near-parallel curves.
def age_effect(age):
    return 6.0 * np.exp(-age / 30.0)

y_thickness_train1 = 12 + age_effect(age_train) + 0.01 * feature_volume_train1 + 0.4 * sex_train + np.random.normal(0, 0.15, n_train)
y_thickness_train2 = 12 + age_effect(age_train) + 0.01 * feature_volume_train2 + 0.4 * sex_train + np.random.normal(0, 0.15, n_train)
y_thickness_train3 = 12 + age_effect(age_train) + 0.01 * feature_volume_train3 + 0.01 * feature_volume_train4 + 0.4 * sex_train + np.random.normal(0, 0.15, n_train)
y_thickness_train4 = 12 + age_effect(age_train) + 0.01 * feature_volume_train5 + 0.4 * sex_train + np.random.normal(0, 0.15, n_train)

train_data = {
    'site': np.random.choice(train_sites, size=n_train),
    'sex': sex_train,
    'age': age_train,
    'feature_volume1': feature_volume_train1,
    'feature_volume2': feature_volume_train2,
    'feature_volume3': feature_volume_train3,
    'feature_volume4': feature_volume_train4,
    'feature_volume5': feature_volume_train5,
    'y_thickness1': y_thickness_train1,
    'y_thickness2': y_thickness_train2,
    'y_thickness3': y_thickness_train3,
    'y_thickness4': y_thickness_train4
}
train_data = pd.DataFrame(train_data)

# Generate test dataset
n_test = 500
morbidity_size = 100
test_sites_seeds = train_sites_seeds + [6, 7, 8]  # Additional fixed seeds for test sites
test_sites = [generate_random_site_id(5, seed) for seed in test_sites_seeds]
age_test = np.random.randint(18, 80, n_test)
sex_test = np.random.choice([0, 1], size=n_test)
feature_volume_test1 = np.random.uniform(800, 1200, n_test)
feature_volume_test2 = np.random.uniform(800, 1200, n_test)
feature_volume_test3 = np.random.uniform(800, 1200, n_test)
feature_volume_test4 = np.random.uniform(800, 1200, n_test)
feature_volume_test5 = np.random.uniform(800, 1200, n_test)
# Same non-linear age trajectory as the training data
y_thickness_test1 = 12 + age_effect(age_test) + 0.01 * feature_volume_test1 + 0.4 * sex_test + np.random.normal(0, 0.15, n_test)
y_thickness_test2 = 12 + age_effect(age_test) + 0.01 * feature_volume_test2 + 0.4 * sex_test + np.random.normal(0, 0.15, n_test)
y_thickness_test3 = 12 + age_effect(age_test) + 0.01 * feature_volume_test3 + 0.01 * feature_volume_test4 + 0.4 * sex_test + np.random.normal(0, 0.15, n_test)
y_thickness_test4 = 12 + age_effect(age_test) + 0.01 * feature_volume_test5 + 0.4 * sex_test + np.random.normal(0, 0.15, n_test)

# Add outliers to the test data
outlier_indices = np.random.choice(n_test, size=morbidity_size, replace=False)
y_thickness_test1[outlier_indices[:(morbidity_size//4)]] += np.random.normal(3, 0.05, morbidity_size//4)  # Positive outliers for y_thickness1
y_thickness_test1[outlier_indices[(morbidity_size//4):(morbidity_size//2)]] -= np.random.normal(3, 0.05, morbidity_size//4)  # Negative outliers for y_thickness1
y_thickness_test2[outlier_indices[(morbidity_size//2):(3*morbidity_size//4)]] += np.random.normal(3, 0.05, morbidity_size//4)  # Positive outliers for y_thickness2
y_thickness_test2[outlier_indices[(3*morbidity_size//4):]] -= np.random.normal(3, 0.05, morbidity_size//4)  # Negative outliers for y_thickness2

morbidities = ['None', 'Schizo', 'Dementia', 'MDD']
affected_test = np.full(n_test, 'None', dtype=object)
affected_test[outlier_indices] = np.random.choice(morbidities[1:], size=morbidity_size)  # Assign morbidities only to outliers

test_data = {
    'site': np.random.choice(test_sites, size=n_test),
    'sex': sex_test,
    'age': age_test,
    'feature_volume1': feature_volume_test1,
    'feature_volume2': feature_volume_test2,
    'feature_volume3': feature_volume_test3,
    'feature_volume4': feature_volume_test4,
    'feature_volume5': feature_volume_test5,
    'y_thickness1': y_thickness_test1,
    'y_thickness2': y_thickness_test2,
    'y_thickness3': y_thickness_test3,
    'y_thickness4': y_thickness_test4,
    'affected': affected_test
}
test_data = pd.DataFrame(test_data)

# Print the datasets
print("Training Dataset:")
display(train_data.head())
print("\nTesting Dataset:")
display(test_data.head())


Training Dataset:


,site,sex,age,feature_volume1,feature_volume2,feature_volume3,feature_volume4,feature_volume5,y_thickness1,y_thickness2,y_thickness3,y_thickness4
0,HJDXM,0,56,848.209032,1019.328233,942.341904,844.454859,1171.438753,21.277471,23.285272,30.851458,24.390216
1,BCCLF,0,69,1092.409469,851.195283,833.567814,1059.242014,1095.497404,23.369578,21.380807,31.456619,23.368378
2,ESZYC,1,46,877.039611,1152.854246,933.385037,1150.047218,922.507155,22.482468,25.267274,34.556324,23.208468
3,BCCLF,0,32,846.112808,957.379139,1071.016378,1116.208204,906.032514,22.553167,23.751988,36.149900,23.229095
4,HSREL,0,60,968.705064,1131.147350,828.844437,856.443038,1078.476980,22.484007,24.163945,29.541937,23.391176



Testing Dataset:


,site,sex,age,feature_volume1,feature_volume2,feature_volume3,feature_volume4,feature_volume5,y_thickness1,y_thickness2,y_thickness3,y_thickness4,affected
0,BCCLF,0,30,1023.903397,848.225798,826.236346,829.913302,1012.805391,24.390944,22.570610,30.784494,24.170846,None
1,HJDXM,0,31,943.319765,847.976074,1082.031971,1166.870174,964.376733,26.720256,22.602273,36.553445,24.038759,Schizo
2,KEMUB,0,61,834.336911,1090.357464,1017.414439,1096.616055,1040.821469,21.166279,23.691480,33.927770,23.309050,None
3,HJDXM,0,57,1091.966252,1027.887763,1010.311723,934.196882,1053.981485,23.772841,23.165027,32.182817,23.922547,None
4,TIXLZ,0,70,1148.455260,1013.496792,916.818223,945.175512,817.113312,23.895126,22.691146,31.215588,20.647750,None
